In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Define paths
data_dir = "labeled_images"
annotation_file = os.path.join(data_dir, "_annotations.coco.json")

# Load COCO annotations
with open(annotation_file, 'r') as f:
    coco_data = json.load(f)

# Process annotations to create a dataset
def process_annotations(coco_data):
    # Create a mapping from image ID to filename
    image_id_to_filename = {img['id']: img['file_name'] for img in coco_data['images']}
    
    # Process annotations and classify images
    image_labels = {}
    
    for annotation in coco_data['annotations']:
        image_id = annotation['image_id']
        filename = image_id_to_filename[image_id]
        
        # Get user tags
        user_tags = annotation.get('user_tags', [])
        
        # Classify the image
        if 'drowsy' in user_tags and 'Alert' in user_tags:
            label = 'Alert'  # Images with both tags are classified as Alert
        elif 'drowsy' in user_tags:
            label = 'Drowsy'  # Images with only drowsy tag
        else:
            continue  # Skip images without relevant tags
        
        image_labels[filename] = label
    
    return image_labels

# Process the annotations
image_labels = process_annotations(coco_data)
print(f"Total labeled images: {len(image_labels)}")

# Display label distribution
label_counts = pd.Series(image_labels.values()).value_counts()
print("Label distribution:")
print(label_counts)

# Define dataset class
class DrowsinessDataset(Dataset):
    def __init__(self, image_labels, data_dir, transform=None):
        self.image_paths = list(image_labels.keys())
        self.labels = [1 if image_labels[img_path] == 'Alert' else 0 for img_path in self.image_paths]
        self.data_dir = data_dir
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_paths[idx])
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Split the dataset
image_paths = list(image_labels.keys())
train_paths, val_paths = train_test_split(image_paths, test_size=0.2, random_state=42, 
                                          stratify=[image_labels[p] for p in image_paths])

# Create dictionaries for training and validation sets
train_labels = {p: image_labels[p] for p in train_paths}
val_labels = {p: image_labels[p] for p in val_paths}

# Create datasets
train_dataset = DrowsinessDataset(train_labels, data_dir, transform=train_transform)
val_dataset = DrowsinessDataset(val_labels, data_dir, transform=val_transform)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# Define the CNN model (using a pre-trained ResNet-18)
def create_model():
    model = models.resnet18(pretrained=True)
    
    # Freeze all parameters
    for param in model.parameters():
        param.requires_grad = False
    
    # Replace the final fully connected layer
    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, 2)  # 2 classes: Drowsy and Alert
    )
    
    return model

# Create model
model = create_model()
model = model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# Define training function
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} (Training)'):
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_acc)
        
        # Validation phase
        model.eval()
        val_running_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} (Validation)'):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_running_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_epoch_loss = val_running_loss / len(val_loader.dataset)
        val_epoch_acc = val_correct / val_total
        val_losses.append(val_epoch_loss)
        val_accuracies.append(val_epoch_acc)
        
        print(f'Epoch {epoch+1}/{num_epochs}:')
        print(f'Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}')
        print(f'Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}')
    
    return train_losses, val_losses, train_accuracies, val_accuracies

# Train the model
num_epochs = 10
train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model, train_loader, val_loader, criterion, optimizer, num_epochs=num_epochs
)

# Plot results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs+1), val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, num_epochs+1), val_accuracies, label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

# Save the model
torch.save(model.state_dict(), 'drowsiness_detection_model.pth')
print("Model saved as 'drowsiness_detection_model.pth'")

KeyboardInterrupt: 